In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import yfinance as yf
import cvxpy as cvx
import riskfolio as rf

import statsmodels.api as sm

import datetime as dt
from arch import arch_model

from statsmodels.graphics.tsaplots import plot_pacf, plot_acf

In [2]:
# 1. Download historical SPY data
# Using yfinance to get data
ticker = 'SPY'
end_date = pd.Timestamp.today()
start_date = end_date - pd.DateOffset(years=5) # Get last 5 years of data
spy_data = yf.download(ticker, start=start_date, end=end_date, auto_adjust=True)

[*********************100%***********************]  1 of 1 completed

1 Failed download:
['SPY']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


In [3]:
spy_date.head()

NameError: name 'spy_date' is not defined

In [6]:
# Ensure data is sorted by date
spy_data.sort_index(inplace=True)

# 2. Calculate log returns (expressed in percentage to help the optimizer converge)
# Drop any missing values created by the shift operation
returns = 100 * spy_data['Adj Close'].pct_change().dropna()

# Plot the returns to observe volatility clustering
# returns.plot(title="SPY Daily Returns (%)")
# plt.show()

# 3. Specify and fit the GARCH(1,1) model
# We use 'GARCH' for volatility model, 'constant' for mean model, 
# and 'Normal' for distribution, with p=1 and q=1
model = arch_model(returns, vol='Garch', p=1, o=0, q=1, mean='constant', dist='normal')
results = model.fit(disp='off') # 'disp="off"' prevents fit summary from printing during fitting

# Print the model summary to see parameters like alpha[1] and beta[1]
print(results.summary())

# 4. Make a 1-step ahead forecast
# The forecast is made using data up to and including the last date in the returns series
forecasts = results.forecast(horizon=1, start=returns.index[-1], method='analytical')

# The output of the forecast is a DataFrame of variances
# Get the variance for the next period (h.1)
# Square root the variance to get the conditional volatility
next_day_variance = forecasts.variance.iloc[-1, 0]
next_day_volatility = np.sqrt(next_day_variance)

print(f"\nLast observation date: {returns.index[-1].date()}")
print(f"Forecasted 1-day ahead conditional volatility for the next trading day: {next_day_volatility:.4f}%")

# To get annualized volatility, multiply by sqrt(252)
annualized_volatility = next_day_volatility * np.sqrt(252)
print(f"Forecasted 1-day ahead annualized volatility: {annualized_volatility:.4f}%")

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed

1 Failed download:
['SPY']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


ValueError: first_obs and last_obs produce an empty array.